# Como as defesas se distribuem entre programas, níveis e modalidades?

**Pergunta 2.** Olhando as bancas de defesa do ICMC, quais combinações de programa, nível (mestrado/doutorado) e modalidade (remoto/presencial) são as mais comuns e quais são as mais raras?

**Dados.** Cada linha é um evento (`data/eventos.csv`). Entram só as bancas do tipo **defesa**; as qualificações ficam de fora para não duplicar o mesmo aluno. A coluna `remoto` (1/0) foi convertida em "Remoto" e "Presencial", e o nome de cada programa foi reduzido à sua sigla. Em seguida, contamos quantas defesas existem para cada combinação de `programa`, `nivel` e `modalidade`.

**Gráfico.** Um **treemap hierárquico interativo**, em que a área de cada caixa é proporcional ao número de defesas:
1. **Hierarquia ajustável:** três seletores permitem escolher qual dimensão fica no nível externo, no médio e no interno. Assim dá para ver os mesmos dados sob perspectivas diferentes (por exemplo, programa → nível → modalidade ou modalidade → programa → nível).
2. **Cores:** cada categoria do nível externo tem sua própria paleta. Nas caixas mais internas, o tom varia conforme a quantidade, e as combinações mais frequentes aparecem mais escuras.

In [82]:
import ipywidgets as widgets
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import pandas as pd
import plotly.express as px
from IPython.display import display

ARQ_EVENTOS = '../../data/eventos_3.csv'
TIPO_DEFESA = 'defesa'
MAPA_MODALIDADE = {1: 'Remoto', 0: 'Presencial'}
COLUNAS_ANALISE = ['programa', 'nivel', 'modalidade']
SEPARADOR_PROGRAMA = ' - '   # o nome do programa é reduzido ao trecho depois do último separador

# Mapeamento amigável para os nomes dos seletores
OPCOES_DIMENSOES = {
    'Programa de Pós-Graduação': 'programa',
    'Nível (Mestrado/Doutorado)': 'nivel',
    'Modalidade (Remoto/Presencial)': 'modalidade',
}

# Paletas do matplotlib, uma para cada categoria do nível externo
CMAPS_DISPONIVEIS = [
    cm.Blues,
    cm.Greens,
    cm.Purples,
    cm.Oranges,
    cm.Reds,
    cm.YlGnBu,
]

# Posição (0 a 1) em cada paleta
COR_EXTERNO = 0.4          # caixa do nível externo
COR_MEDIO = 0.55           # caixa do nível médio (um pouco mais escura)
COR_FOLHA_MIN = 0.3        # folha com a menor quantidade
COR_FOLHA_AMPLITUDE = 0.6  # quanto o tom sobe até a folha com a maior quantidade
COR_FOLHA_UNICA = 0.7      # quando todas as folhas têm a mesma quantidade

LARGURA, ALTURA = 1100, 650
TAMANHO_MIN_TEXTO = 10

## 1. Dados

Carregamos os eventos e ficamos só com as **defesas**. A coluna `remoto` (1 ou 0) vira `modalidade` ("Remoto" ou "Presencial"), que fica mais legível nos rótulos do gráfico.

In [83]:
df = pd.read_csv(ARQ_EVENTOS)

# Filtrando apenas as Defesas (removendo qualificações para evitar duplicação)
df_defesa = df[df['tipo'].str.lower() == TIPO_DEFESA].copy()

# Mapeando a coluna 'remoto' para termos descritivos legíveis
df_defesa['modalidade'] = df_defesa['remoto'].map(MAPA_MODALIDADE)

df_defesa[COLUNAS_ANALISE].head()   # (opcional) só para conferir o resultado

,programa,nivel,modalidade
0,Ciências de Computação e Matemática Computacio...,Doutorado,Remoto
4,Matemática em Rede Nacional - PROFMAT,Mestrado,Remoto
6,Matemática - PPG-Mat,Doutorado,Remoto
8,Ciências de Computação e Matemática Computacio...,Doutorado,Presencial
10,Matemática - PPG-Mat,Mestrado,Remoto


## 2. Combinações mais comuns e mais raras

Contamos quantas defesas existem em cada combinação de `programa`, `nivel` e `modalidade`. Abaixo estão as 3 combinações mais raras e as 3 mais comuns, para termos uma ideia do que esperar do gráfico.

Aqui o programa aparece com o nome completo. No treemap ele é reduzido à sigla, para caber nos rótulos.

In [84]:
# Agrupando os dados para contar as ocorrências de cada combinação completa
df_contagem = (
    df_defesa.groupby(COLUNAS_ANALISE).size().reset_index(name='quantidade')
)

# 1. Obtendo os mais raros (ordenação crescente)
df_raros = df_contagem.sort_values(by='quantidade', ascending=True).reset_index(
    drop=True
)

# 2. Obtendo os mais comuns (ordenação decrescente)
df_comuns = df_contagem.sort_values(
    by='quantidade', ascending=False
).reset_index(drop=True)

# Exibindo os resultados
print('=== OS 3 PERFIS DE BANCA MAIS RAROS (Menor Frequência) ===')
display(df_raros.head(3))

print('\n=== OS 3 PERFIS DE BANCA MAIS COMUNS (Maior Frequência) ===')
display(df_comuns.head(3))

print('\nDataset pronto para a visualização interativa!')

=== OS 3 PERFIS DE BANCA MAIS RAROS (Menor Frequência) ===


,programa,nivel,modalidade,quantidade
0,Matemática em Rede Nacional - PROFMAT,Mestrado,Presencial,5
1,Estatística - PIPGES,Doutorado,Presencial,7
2,Matemática - PPG-Mat,Mestrado,Remoto,9



=== OS 3 PERFIS DE BANCA MAIS COMUNS (Maior Frequência) ===


,programa,nivel,modalidade,quantidade
0,Ciências de Computação e Matemática Computacio...,Mestrado,Remoto,125
1,Ciências de Computação e Matemática Computacio...,Doutorado,Remoto,99
2,Ciências de Computação e Matemática Computacio...,Doutorado,Presencial,64



Dataset pronto para a visualização interativa!


## 3. Gráfico: treemap hierárquico

Cada caixa é um grupo de defesas e a **área é proporcional à quantidade**. As caixas ficam aninhadas na ordem escolhida: a externa contém as do meio, que contêm as internas.

- Os três seletores definem a hierarquia (por exemplo, programa → nível → modalidade ou modalidade → programa → nível). Cada dimensão só pode aparecer uma vez; se repetir, o gráfico mostra um aviso.
- Cada categoria do nível externo tem sua própria cor. Nas caixas mais internas, **quanto mais escuro o tom, mais defesas** dentro do mesmo grupo externo.
- Passe o mouse numa caixa para ver a quantidade. Clique nela para dar zoom no grupo e clique na faixa de cima para voltar.

In [85]:
# Rótulos das dimensões, exibidos no hover (as que faltarem usam o nome da coluna)
ROTULOS_DIM = {
    'programa': 'Programa',
    'nivel': 'Nível',
    'modalidade': 'Modalidade',
}
MAX_IRMAOS = 8  # máximo de caixas irmãs com valor listadas no hover
MAX_ZEROS = 5   # máximo de caixas irmãs zeradas listadas no hover


def fmt_pct(x):
  return f'{100 * x:.1f}'.replace('.', ',') + '%'


def fmt_defesas(n):
  return f'{n} defesa' if n == 1 else f'{n} defesas'


def cor_texto(hex_cor):
  """Branco em fundo escuro, quase preto em fundo claro."""
  r, g, b = mcolors.to_rgb(hex_cor)
  return '#1a1a1a' if 0.299 * r + 0.587 * g + 0.114 * b > 0.6 else 'white'


def escurecer(hex_cor, fator=0.6):
  r, g, b = mcolors.to_rgb(hex_cor)
  return mcolors.to_hex((r * fator, g * fator, b * fator))


def montar_hover(path, dims, valor, valor_pai, total, irmaos):
  """Texto HTML do hover de uma caixa.

  path: rótulos do topo até a caixa (ex.: ['PPG-Mat', 'Mestrado', 'Remoto'])
  dims: nomes das dimensões desse mesmo caminho
  valor_pai: quantidade da caixa pai (None no nível externo)
  irmaos: {categoria: quantidade} de TODAS as categorias daquela dimensão,
          inclusive as que valem 0 (e por isso não têm retângulo)
  """
  atual = path[-1]
  linhas = [f'<b>{" › ".join(str(p) for p in path)}</b>']

  # Quantidade exata e porcentagens
  partes = [fmt_defesas(valor)]
  if valor_pai is not None:
    partes.append(f'{fmt_pct(valor / valor_pai)} do grupo')
  partes.append(f'{fmt_pct(valor / total)} do total')
  linhas.append(' · '.join(partes))

  # Posição entre as caixas irmãs e resumo delas (incluindo os zeros)
  if len(irmaos) > 1:
    posicao = 1 + sum(1 for v in irmaos.values() if v > valor)
    empate = sum(1 for v in irmaos.values() if v == valor) > 1
    rotulo = ROTULOS_DIM.get(dims[-1], dims[-1].capitalize())
    linhas.append(
        f'{rotulo} — {posicao}ª maior de {len(irmaos)}'
        + (' (empate)' if empate else '')
    )

    ordenados = sorted(irmaos.items(), key=lambda kv: (-kv[1], str(kv[0])))
    com_valor = [(c, v) for c, v in ordenados if v > 0]
    zeros = [str(c) for c, v in ordenados if v == 0]

    mostrados = com_valor[:MAX_IRMAOS]
    if all(c != atual for c, _ in mostrados):  # garante que a própria caixa apareça
      mostrados = mostrados[:-1] + [(atual, valor)]
    itens = [f'<b>{c}: {v}</b>' if c == atual else f'{c}: {v}' for c, v in mostrados]
    if len(com_valor) > len(mostrados):
      itens.append(f'… +{len(com_valor) - len(mostrados)}')
    linhas.append(' · '.join(itens))

    if zeros:
      txt = ', '.join(zeros[:MAX_ZEROS])
      if len(zeros) > MAX_ZEROS:
        txt += f' … +{len(zeros) - MAX_ZEROS}'
      linhas.append(f'Nenhuma defesa: {txt}')

  return '<br>'.join(linhas)


# Função que atualiza o gráfico dinamicamente com base nas escolhas do usuário
def atualizar_treemap(dim1, dim2, dim3):
  caminho = [dim1, dim2, dim3]

  # Validação para evitar dimensões repetidas na mesma hierarquia
  if len(set(caminho)) < 3:
    print(
        '⚠️ Aviso: Por favor, selecione três dimensões diferentes para cada'
        ' nível da hierarquia.'
    )
    return

  # Agrupamento base com as 3 dimensões
  df_raw = (
      df_defesa.assign(
          programa=df_defesa['programa'].str.rsplit(SEPARADOR_PROGRAMA, n=1).str[-1].str.strip()
      )
      .groupby([dim1, dim2, dim3])
      .size()
      .reset_index(name='quantidade')
  )

  # Totais e categorias usados no hover (porcentagens, posição e caixas zeradas)
  total_geral = df_raw['quantidade'].sum()
  categorias = {d: sorted(df_raw[d].unique()) for d in caminho}
  tot1 = df_raw.groupby(dim1)['quantidade'].sum().to_dict()
  tot2 = df_raw.groupby([dim1, dim2])['quantidade'].sum().to_dict()
  tot3 = df_raw.set_index([dim1, dim2, dim3])['quantidade'].to_dict()

  # Paleta do matplotlib para cada categoria do nível externo (dim1)
  grupos_dim1 = df_raw[dim1].unique()
  mapa_cmaps = {
      g: CMAPS_DISPONIVEIS[i % len(CMAPS_DISPONIVEIS)]
      for i, g in enumerate(grupos_dim1)
  }

  # Listas para construir o DataFrame hierárquico explícito (pais, filhos e nós)
  ids, labels, parents, values, cores_hex, hovers = [], [], [], [], [], []

  # 1. Nível 1: Categoria mais externa (dim1)
  for g in grupos_dim1:
    sub_df = df_raw[df_raw[dim1] == g]
    val_g = sub_df['quantidade'].sum()

    id_1 = str(g)
    ids.append(id_1)
    labels.append(str(g))
    parents.append('')  # Raiz
    values.append(val_g)

    cmap = mapa_cmaps[g]
    # Cor base intermediária para a caixa externa
    cores_hex.append(mcolors.to_hex(cmap(COR_EXTERNO)))
    hovers.append(montar_hover(
        [g], [dim1], val_g, None, total_geral,
        {c: tot1.get(c, 0) for c in categorias[dim1]},
    ))

    # 2. Nível 2: Categoria intermediária (dim2)
    sub_grupos_dim2 = sub_df[dim2].unique()
    for sg2 in sub_grupos_dim2:
      sub2_df = sub_df[sub_df[dim2] == sg2]
      val_sg2 = sub2_df['quantidade'].sum()

      id_2 = f'{id_1} / {sg2}'
      ids.append(id_2)
      labels.append(str(sg2))
      parents.append(id_1)
      values.append(val_sg2)
      # Tom um pouco mais escuro para o nível intermediário
      cores_hex.append(mcolors.to_hex(cmap(COR_MEDIO)))
      hovers.append(montar_hover(
          [g, sg2], [dim1, dim2], val_sg2, val_g, total_geral,
          {c: tot2.get((g, c), 0) for c in categorias[dim2]},
      ))

      # 3. Nível 3: Folhas/Caixas mais internas (dim3) com gradiente baseado na quantidade
      min_val = sub_df['quantidade'].min()
      max_val = sub_df['quantidade'].max()

      for _, row in sub2_df.iterrows():
        sg3 = row[dim3]
        qtd = row['quantidade']

        id_3 = f'{id_2} / {sg3}'
        ids.append(id_3)
        labels.append(str(sg3))
        parents.append(id_2)
        values.append(qtd)

        # Gradiente interno proporcional à quantidade
        if max_val == min_val:
          norm_val = COR_FOLHA_UNICA
        else:
          norm_val = COR_FOLHA_MIN + COR_FOLHA_AMPLITUDE * (qtd - min_val) / (max_val - min_val)
        cores_hex.append(mcolors.to_hex(cmap(norm_val)))
        hovers.append(montar_hover(
            [g, sg2, sg3], caminho, qtd, val_sg2, total_geral,
            {c: tot3.get((g, sg2, c), 0) for c in categorias[dim3]},
        ))

  # Montando o DataFrame final com a estrutura completa da árvore
  df_tree = pd.DataFrame({
      'ids': ids,
      'labels': labels,
      'parents': parents,
      'values': values,
      'cor_hex': cores_hex,
      'hover': hovers,
      'cor_texto': [cor_texto(c) for c in cores_hex],
      'cor_borda': [escurecer(c) for c in cores_hex],
  })

  dicionario_cores = {c: c for c in df_tree['cor_hex'].unique()}

  # Criando o Treemap interativo
  fig = px.treemap(
      df_tree,
      ids='ids',
      names='labels',
      parents='parents',
      values='values',
      branchvalues='total',
      color='cor_hex',
      color_discrete_map=dicionario_cores,
      custom_data=['hover'],
      title=(
          '<b>Treemap Hierárquico Completo</b><br><sub>Hierarquia atual:'
          f' {" -> ".join(caminho)} | Cores customizadas por nível</sub>'
      ),
  )

  # Hover com fundo na cor da própria caixa; o <extra> remove a caixinha secundária
  fig.update_traces(
      hovertemplate='%{customdata[0]}<extra></extra>',
      hoverlabel=dict(
          bgcolor=df_tree['cor_hex'].tolist(),
          bordercolor=df_tree['cor_borda'].tolist(),
          font=dict(size=14, color=df_tree['cor_texto'].tolist()),
          align='left',
      ),
  )

  fig.update_layout(
      margin=dict(t=70, l=20, r=20, b=20),
      width=LARGURA,
      height=ALTURA,
      uniformtext=dict(minsize=TAMANHO_MIN_TEXTO, mode='hide'),
  )
  fig.show()

In [86]:
# Criando os dropdowns para o usuário escolher a ordem de aninhamento
dropdown_nivel1 = widgets.Dropdown(
    options=OPCOES_DIMENSOES, value='programa', description='1º (Externo):'
)
dropdown_nivel2 = widgets.Dropdown(
    options=OPCOES_DIMENSOES, value='nivel', description='2º (Médio):'
)
dropdown_nivel3 = widgets.Dropdown(
    options=OPCOES_DIMENSOES, value='modalidade', description='3º (Interno):'
)

# Organizando a interface visual com widgets do Jupyter
ui = widgets.VBox(
    [
        widgets.HTML(
            '<b>Escolha a ordem de visualização (todas as caixas herdarão a'
            ' harmonia de cor do primeiro nível):</b>'
        ),
        widgets.HBox([dropdown_nivel1, dropdown_nivel2, dropdown_nivel3]),
    ]
)

# Conectando os seletores à função de atualização
out = widgets.interactive_output(
    atualizar_treemap,
    {'dim1': dropdown_nivel1, 'dim2': dropdown_nivel2, 'dim3': dropdown_nivel3},
)

display(ui, out)

Output()

## 4. Limitações

- **A comparação precisa é difícil entre áreas parecidas.** Como a quantidade é representada pela área, caixas com valores próximos parecem do mesmo tamanho e fica difícil dizer qual é maior só olhando.
- **Caixas pequenas perdem o rótulo.** Combinações com poucas defesas viram caixas pequenas e o texto some, o que dificulta a interpretação à primeira vista. O valor continua aparecendo ao passar o mouse.
- **Zeros não são representados.** Uma combinação sem nenhuma defesa não gera retângulo. Quem lê precisa perceber que a caixa não existe e inferir que o valor é zero.